# Measuring

Section 13.5 of the notes.

Every claim about speed has to be measured. This notebook shows which clock to
use, how to repeat a measurement, how to find the slow line of a function, and
which numbers to distrust.

## 1. Which clock

`time.time` returns the current date and time, which the operating system may
adjust while the program runs. `time.perf_counter` is made for measuring short
intervals: it never runs backwards and has a much finer resolution.

In [ ]:
import time

for name in ("time", "perf_counter", "process_time"):
    info = time.get_clock_info(name)
    print(f"{name:14s} resolution {info.resolution:.0e} s  monotonic {info.monotonic}")

In [ ]:
def smallest_step(clock, tries=200_000):
    # smallest non-zero difference the clock can report
    step = float("inf")
    previous = clock()
    for _ in range(tries):
        now = clock()
        if now > previous:
            step = min(step, now - previous)
            previous = now
    return step


print(f"time.time()          steps by at least {smallest_step(time.time):.2e} s")
print(
    f"time.perf_counter()  steps by at least {smallest_step(time.perf_counter):.2e} s"
)

## 2. Repeat the measurement

A single call is disturbed by everything else the machine is doing. `%timeit`
repeats the call many times and reports the mean and the spread over several
runs, using `perf_counter` internally. The smallest of the runs is the time with
the least disturbance; `%timeit -o` returns the result, and its `.best` is that
smallest time.

In [ ]:
import numpy as np

rng = np.random.default_rng(0)
a = rng.random(1_000_000)  # positive numbers, so sqrt is defined

t = %timeit -o -r 5 np.sqrt(a)
print(f"smallest run: {1e6 * t.best:.0f} us per call")

## 3. The first call is different

The first call of an operation pays for setting up the memory of its result:
the operating system hands over fresh memory page by page as it is first
written. Later calls reuse memory that the previous call has just released, and
are faster. This is why a benchmark discards the first call. The cell times one
call of `np.sqrt` on 20 million entries, the first and then the third.

In [ ]:
big = rng.random(20_000_000)  # 160 MB

t0 = time.perf_counter()
_ = np.sqrt(big)
t1 = time.perf_counter()
_ = np.sqrt(big)
t2 = time.perf_counter()
_ = np.sqrt(big)
t3 = time.perf_counter()
print(f"first call {1e3 * (t1 - t0):6.1f} ms")
print(f"third call {1e3 * (t3 - t2):6.1f} ms")

## 4. Which line is slow

`%prun` counts time per function, which is too coarse when everything happens
inside one function. `%lprun` from `line_profiler` reports the time of each
line.

In [ ]:
%load_ext line_profiler


def pipeline(n=2_000_000):
    a = rng.standard_normal(n)
    b = rng.standard_normal(n)
    total = 0.0
    for i in range(0, n, 10):  # Python loop over 200,000 entries
        total += a[i] * b[i]  # one entry per pass
    y = (a + b) * (a - b)
    return total + y.sum()


%lprun -u 1e-3 -f pipeline pipeline()

The two lines of the Python loop take most of the time, although the loop
touches only one entry in ten. The vectorized line `y = (a + b) * (a - b)`
handles all $2\,000\,000$ entries and takes a small share. Vectorizing the loop
is the only change worth making.

## 5. Numbers to distrust

* **A timing at one size only.** The cost per entry changes with the size of
  the arrays, so a number measured on small arrays says little about the size
  the application uses.
* **Timings that include the first call**, which contains the setting up of
  memory.
* **Totals without a size.** The useful number is the cost per entry, because
  it shows how the method behaves as the problem grows.
* **Speed-ups measured against a weak baseline.** Replacing a pure Python loop
  by a compiled kernel may give a speed-up of $500$. Against the one-line NumPy
  version, `a * b + c`, the same kernel may be only $1.5$ times faster. The
  first number mostly measures how slow the loop is. A fair speed-up compares
  against the best simple alternative, usually the plain NumPy version.

The sweep below shows the first point for the dot product `x @ y`.

In [ ]:
import timeit

# cost per entry of one array operation, over six sizes
for n in (10**3, 10**4, 10**5, 10**6, 10**7, 10**8):
    x, y = rng.standard_normal(n), rng.standard_normal(n)
    best = min(timeit.repeat(lambda: x @ y, repeat=5, number=3)) / 3
    print(f"n = {n:>11,}: {1e9 * best / n:6.3f} ns per entry")

Three regimes appear.

* **Up to $10^4$ entries.** The fixed cost of the call, which does not depend
  on the size, is spread over few entries and dominates the time per entry.
* **$10^5$ entries.** The two vectors take $1.6$ MB and fit in the cache, the
  small fast memory next to the processor. The time per entry is smallest
  here.
* **$10^6$ entries and more.** The vectors no longer fit in the cache and every
  entry comes from main memory, which is slower. The time per entry grows by a
  factor of about four.

The same operation therefore has very different costs per entry depending on
the size of the arrays.